# Tokenizing sentences with Keras

Here you will get your hands dirty with the Keras Tokenizer. The Keras Tokenizer is a great utility that helps you to do some crucial text processing with a few lines of code. For example, the Keras Tokenizer will automatically map the words in your vocabulary to IDs with a single function call. Here, you will learn about this in more detail.

In [1]:
import pandas as pd
en_text = pd.read_csv('dataset/vocab_en.txt', header=None, delimiter='\n')
# en_text.head()
fr_text = pd.read_csv('dataset/vocab_fr.txt', header=None, delimiter='\n')
# fr_text.head()
en_text = en_text.iloc[:,0].values.tolist()
fr_text = fr_text.iloc[:,0].values.tolist()


In [2]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Define a Keras Tokenizer
en_tok = Tokenizer()

# Fit the tokenizer on some text
en_tok.fit_on_texts(en_text)

for w in ["january", "apples", "summer"]:
  # Get the word ID of word w
  id = en_tok.word_index[w]
  # Print the word and the word ID
  print(w, " has id: ", id)

january  has id:  36
apples  has id:  75
summer  has id:  46


# Controlling the vocabulary with the Tokenizer

Let's drill down a bit more into the operation of the Tokenizer. In this exercise you will learn how to convert an arbitrary sentence to a sequence using a trained Tokenizer. Furthermore, you will learn to control the size of the vocabulary of the Tokenizer. You will also investigate what happens to the out-of-vocabulary (OOV) words when you limit the vocabulary size of a Tokenizer.

In [3]:
# Convert the sentence to a word ID sequence
seq = en_tok.texts_to_sequences(['she likes grapefruit , peaches , and lemons .'])
print('Word ID sequence: ', seq)

# Define a tokenizer with vocabulary size 50 and oov_token 'UNK'
en_tok_new = Tokenizer(num_words=100, oov_token='UNK') 

# Fit the tokenizer on en_text
en_tok_new.fit_on_texts(en_text)

# Convert the sentence to a word ID sequence
seq_new = en_tok_new.texts_to_sequences(['she likes grapefruit , peaches , and lemons .'])
print('Word ID sequence (with UNK): ', seq_new)
print('The ID 1 represents the word: ', en_tok_new.index_word[1])

Word ID sequence:  [[27, 70, 28, 76, 7, 72]]
Word ID sequence (with UNK):  [[28, 71, 29, 77, 8, 73]]
The ID 1 represents the word:  UNK


# Adding special tokens

You will now learn to add sos (marks the start) and eos (marks the end) tokens to the sentences. As already discussed, this step is optional for the model you have right now, but these will be required for a model that you'll be implementing in a later chapter.

In [4]:
fr_text_new = []

# Loop through all sentences in fr_text
for sent in fr_text:
  
#   print("Before adding tokens: ", sent)
  
  # Add sos and eos tokens using string.join
  sent_new = " ".join(["sos", sent, "eos"])
  # Append the modified sentence to fr_text_new
  fr_text_new.append(sent_new)
  
  # Print sentence after adding tokens
#   print("After adding tokens: ", sent_new, '\n')

# Padding sentences

You will now implement a function called `sents2seqs()` which you will later use to transform data conveniently to the format accepted by the neural machine translation (NMT) model. sents2seqs() accepts a list of sentence strings and,

- Converts the sentences to a list of sequence of IDs,
- Pad the sentences so that they have equal length and,
- Optionally convert the IDs to onehot vectors.

In [5]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

en_len = 15
en_vocab = 100 # 100
en_tok = en_tok_new
def sents2seqs(input_type, sentences, onehot=False, pad_type='post'):
	# Convert sentences to sequences      
    encoded_text = en_tok.texts_to_sequences(sentences)
    # Pad sentences to en_len
    preproc_text = pad_sequences(encoded_text, padding=pad_type, truncating="post", maxlen=en_len)
    if onehot:
		# Convert the word IDs to onehot vectors
        preproc_text = to_categorical(preproc_text, num_classes=en_vocab)
    return preproc_text
sentence = ['she likes grapefruit , peaches , and lemons .'  ]
# Convert a sentence to sequence by pre-padding the sentence
pad_seq = sents2seqs('source', sentence, pad_type="pre")
pad_seq

array([[ 0,  0,  0,  0,  0,  0,  0,  0,  0, 28, 71, 29, 77,  8, 73]])

# Reversing sentences

Here you will learn how to reverse sentences for the encoder model. As discussed, reversing source sentences helps to form a strong initial connection between the encoder and the decoder, which boosts the performance of the model. However, always remember that the benefit is dependent on the two languages you are translating between. As long as they have the same subject, verb ,and object order, it will benefit the model.

In [6]:
sentences = ["california is never rainy during july ."]
# Add new keyword parameter reverse which defaults to False
def sents2seqs(input_type, sentences, onehot=False, pad_type='post', reverse=False):     
    encoded_text = en_tok.texts_to_sequences(sentences)
    preproc_text = pad_sequences(encoded_text, padding=pad_type, truncating='post', maxlen=en_len)
    if reverse:
      # Reverse the text using numpy axis reversing
      preproc_text = preproc_text[:, ::-1]
    if onehot:
        preproc_text = to_categorical(preproc_text, num_classes=en_vocab) # -1??
    return preproc_text
# Call sents2seqs to get the padded and reversed sequence of IDs
pad_seq = sents2seqs('source', sentences, reverse= True)
rev_sent = [en_tok.index_word[wid] for wid in pad_seq[0][-6:]] 
print('\tReversed: ',' '.join(rev_sent))

	Reversed:  july during rainy never is california


# Training the model

You will train the previously implemented model in this exercise. Do you know that the Google's encoder-decoder based machine translation model took 2-4 days to train?

For this exercise you will be using a small dataset of 1500 sentences (i.e. en_text and fr_text) to train the model. This amount will hardly be enough to see good performance, but the method will remain the same. It is a matter of training on more data for longer. You have also been provided with the model 

In [7]:
from keras.layers import Input, GRU, RepeatVector, TimeDistributed, Dense
from keras.models import Model

# Define input shape
input_shape = (en_len, en_vocab)  # Assuming input sequence length is 15 and embedding dimension is 100

# Define encoder
encoder_input = Input(shape=input_shape, name='input_1')
encoder_gru = GRU(48, name='gru', return_state=True)(encoder_input)
encoder_output, encoder_state = encoder_gru

# Define decoder
decoder_input = RepeatVector(15, name='repeat_vector')(encoder_state)
decoder_gru = GRU(48, return_sequences=True, name='gru_1')(decoder_input, initial_state=encoder_state)
decoder_output = TimeDistributed(Dense(en_vocab, activation='softmax'), name='time_distributed')(decoder_gru)

# Create model
nmt = Model(inputs=encoder_input, outputs=decoder_output)

# Print model summary
nmt.summary()


# nmt.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])

Model: "functional_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 15, 100)]    0                                            
__________________________________________________________________________________________________
gru (GRU)                       [(None, 48), (None,  21600       input_1[0][0]                    
__________________________________________________________________________________________________
repeat_vector (RepeatVector)    (None, 15, 48)       0           gru[0][1]                        
__________________________________________________________________________________________________
gru_1 (GRU)                     (None, 15, 48)       14112       repeat_vector[0][0]              
                                                                 gru[0][1]             

In [8]:
# making sure that all the text are in 15 characters of length

def sentence_resizer(text):
    new_text = []
    for sentence in text:
        if len(sentence) > 15:
            text = sentence[:15]
            new_text.append(text)
        else:
            new_text.append(sentence)
    return new_text
        
en_text = sentence_resizer(en_text)
fr_text = sentence_resizer(fr_text)

for sentence in fr_text:
    if len(sentence) > 15:
        print(en_text)
        
nmt.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])
print(len(en_text))
print(len(fr_text))
print(len(fr_text[0]))

137860
137860
15


In [10]:
n_epochs, bsize = 3, 250
data_size = 1500

for ei in range(n_epochs):
  for i in range(0,data_size,bsize):
    # Get a single batch of encoder inputs
    en_x = sents2seqs('source', en_text[i:i+bsize], onehot=True, reverse=True)
    # print(en_x.shape)
    # Get a single batch of decoder outputs
    de_y = sents2seqs('target', fr_text[i:i+bsize], onehot=True)
    
    # Train the model on a single batch of data
    nmt.train_on_batch(en_x, de_y)    
    # Obtain the eval metrics for the training data
    res = nmt.evaluate(en_x, de_y, batch_size=bsize, verbose=0)
    print("{} => Train Loss:{}, Train Acc: {}".format(ei+1,res[0], res[1]*100.0))  


1 => Train Loss:3.411015510559082, Train Acc: 78.37333083152771
1 => Train Loss:3.2135603427886963, Train Acc: 77.97333598136902
1 => Train Loss:2.9835634231567383, Train Acc: 78.24000120162964
1 => Train Loss:2.7481186389923096, Train Acc: 77.99999713897705
1 => Train Loss:2.4905221462249756, Train Acc: 77.84000039100647
1 => Train Loss:2.225759744644165, Train Acc: 77.78666615486145
2 => Train Loss:1.9603339433670044, Train Acc: 78.37333083152771
2 => Train Loss:1.7445130348205566, Train Acc: 77.97333598136902
2 => Train Loss:1.5474653244018555, Train Acc: 78.24000120162964
2 => Train Loss:1.4009777307510376, Train Acc: 77.99999713897705
2 => Train Loss:1.2842758893966675, Train Acc: 77.84000039100647
2 => Train Loss:1.1972556114196777, Train Acc: 77.78666615486145
3 => Train Loss:1.1031062602996826, Train Acc: 78.37333083152771
3 => Train Loss:1.0725007057189941, Train Acc: 77.97333598136902
3 => Train Loss:1.0260674953460693, Train Acc: 78.24000120162964
3 => Train Loss:0.993712842

# Splitting data to training and validation sets

You learned that using only the training data without a validation dataset leads to a problem called overfitting. When overfitting occurs, the model will be very good at predicting data for training inputs, however generalize very poorly to unseen data. This means the model will not be very useful, as it cannot generalize. To avoid this you can use a validation dataset.

In [11]:
import numpy as np
train_size, valid_size = 800, 200
# Define a sequence of indices from 0 to len(en_text)
inds = np.arange(len(en_text))
np.random.shuffle(inds)
train_inds = inds[:train_size]
# Define valid_inds: last valid_size indices
valid_inds = inds[train_size:train_size+valid_size]
# Define tr_en (train EN sentences) and tr_fr (train FR sentences)
tr_en = [en_text[ti] for ti in train_inds]
tr_fr = [fr_text[ti] for ti in train_inds]
# Define v_en (valid EN sentences) and v_fr (valid FR sentences)
v_en = [en_text[vi] for vi in valid_inds]
v_fr = [fr_text[vi] for vi in valid_inds]
print('Training (EN):\n', tr_en[:3], '\nTraining (FR):\n', tr_fr[:3])
print('\nValid (EN):\n', v_en[:3], '\nValid (FR):\n', v_fr[:3])

Training (EN):
 ['your least favo', 'the mango is ou', 'paris is someti'] 
Training (FR):
 ['votre fruit pré', 'la mangue est n', 'paris est parfo']

Valid (EN):
 ['france is never', 'his least liked', 'she dislikes ma'] 
Valid (FR):
 ['la france est j', 'son fruit est m', 'elle déteste le']


# Training the model with validation

Here you will learn how to train the neural machine translator model with a validation step.

You are provided with the nmt model that you created in the last chapter. Furthermore, you will train the model on a English and French sentences obtained from the Udacity Github Repo. You are provided with training English text (tr_en) and French text (tf_fr) as well as validation English text (v_en) and French text (v_fr) from the previous exercise.

In [12]:
# Convert validation data to onehot
v_en_x = sents2seqs('source', en_text, onehot=True, reverse=True)
v_de_y = sents2seqs('target', fr_text, onehot=True)

n_epochs, bsize = 3, 250
for ei in range(n_epochs):
  for i in range(0,train_size,bsize):
    # Get a single batch of inputs and outputs
    en_x = sents2seqs('source', tr_en[i:i+bsize], onehot=True, reverse=True)
    de_y = sents2seqs('target', tr_fr[i:i+bsize], onehot=True)
    # Train the model on a single batch of data
    nmt.train_on_batch(en_x, de_y)    
  # Evaluate the trained model on the validation data
  res = nmt.evaluate(v_en_x, v_de_y, batch_size=valid_size, verbose=0)
  print("{} => Loss:{}, Val Acc: {}".format(ei+1,res[0], res[1]*100.0))

1 => Loss:0.8013154864311218, Val Acc: 78.04052233695984
2 => Loss:0.6853416562080383, Val Acc: 78.04052233695984
3 => Loss:0.5925396084785461, Val Acc: 83.42733979225159


# Part 1: Treasure hunt

You recently won a all-paid trip to a lush tropical island. While you were wandering around, you found an ancient treasure map pointing to a great treasure, which had a few secret messages written using 1s and 0s. Having just taken this course, you instantly recognize that it is a sequence of onehot encoded vectors. You have also been lucky to find the word to index mapping to know which word refers to which ID.

In [13]:
treasure_map = ""
# # Get the word IDs from the treasure map
# word_ids = np.argmax(treasure_map, axis=-1)
# # Get the sequence length from the treasure map
# seq_len = treasure_map.shape[1]

# for i in range(treasure_map.shape[0]):
# 	words = []
# 	for t in range(seq_len):
#       	# Get the word ID for the i-th sentence and t-th position
# 	    wid = word_ids[i, t]
# 	    if wid != 0:
#           	# Append the word corresponding to wid
# 	        words.append(en_tok.index_word[wid])
# 	print("Instruction ", i+1, ": ", ' '.join(words))

# Part 2: Treasure hunt

Now there's a little twist to the treasure hunt. You have forgotten to pack your laptop and you only have a device with limited memory on you. The code you write should be less than 4 lines of code (excluding comments). Since you need to make the code as compact as possible you will be using list comprehension.

In [14]:
# # Get the word IDs from the treasure map
# word_ids = np.argmax(treasure_map, axis=-1)
# # Get the batch size from the treasure map
# for i in range(treasure_map.shape[0]):
#   	# Get all the words of the i-th sentence using list comprehension
# 	words = [en_tok.index_word[wid] for wid in word_ids[i] if wid != 0]
# 	print("Instruction ", i+1, ": ", ' '.join(words))

# Generating English-French translations

Did you know that HSBC bank once spent $10 million on re-branding its slogan, due to a translation mistake?

We will use the trained model to predict the French translation of an English sentence using model.predict(). 

In [15]:
model = nmt

In [16]:
en_st = ['the united states is sometimes chilly during december , but it is sometimes freezing in june .']
print('English: {}'.format(en_st))

# Convert the English sentence to a sequence
en_seq = sents2seqs('source', en_st, onehot=True, reverse=True)

# Predict probabilities of words using en_seq
fr_pred = model.predict(en_seq)

# Get the sequence indices (max argument) of fr_pred
fr_seq = np.argmax(fr_pred, axis=-1)[0]

# Convert the sequence of IDs to a sentence and print
fr_sent = [en_tok.index_word[i] for i in fr_seq if i != 0]
print("French (Custom): {}".format(' '.join(fr_sent)))
print("French (Google Translate): les etats-unis sont parfois froids en décembre, mais parfois gelés en juin")

English: ['the united states is sometimes chilly during december , but it is sometimes freezing in june .']
French (Custom): 
French (Google Translate): les etats-unis sont parfois froids en décembre, mais parfois gelés en juin
